# ML Modelling — When RMSE and R² Disagree With MAE (Exoplanet Orbital Periods)

## Problem

**Dataset:** [NASA Exoplanet Archive — confirmed planets](https://exoplanetarchive.ipac.caltech.edu/) (1,035 real confirmed exoplanets as of the catalog snapshot), loaded from the `mwaskom/seaborn-data` mirror (`planets.csv`): `method` (how the planet was detected), `number` (planets in that system), `orbital_period` (days), `mass` (Jupiter masses), `distance` (parsecs from Earth), `year` (year of discovery).

This dataset has two real, unforgiving data-quality problems baked in, not injected:

- **Heavy missingness**: `mass` is missing for more than half the rows; `distance` is missing for about a fifth.
- **An extreme heavy tail**: `orbital_period` ranges from 0.09 days to over 700,000 days — a handful of ultra-long-period planets are four to five orders of magnitude larger than the typical planet.

That tail is what makes this a good ML-quality exercise: a single outlier can make RMSE and R² look terrible while MAE looks fine (or vice versa), and which story you believe changes what you'd tell a stakeholder about whether the model is any good.

## What you should produce

Fill in the TODOs in the starter cells below to:

1. **`clean_planets(df)`** — apply and *document* a defensible missing-data strategy: drop rows missing the target (`orbital_period`) or missing `distance`; drop the `mass` **column** entirely (don't impute it — explain in your write-up why imputing a column that's >50% missing would be misleading rather than helpful). Return `(cleaned_df, audit)` where `audit` is a dict recording how many rows were dropped at each step.
2. **`build_features(df)`** — build a feature matrix from `distance`, `year`, and `method` bucketed into the top 3 methods plus `"Other"` (one-hot encoded), and a target of `log1p(orbital_period)` (the log transform is because of the heavy tail — you'll see what happens if you skip it).
3. **`evaluate_regression(y_true_log, y_pred_log)`** — back-transform predictions with `expm1`, clip negative predictions to 0, and return a dict of **MAE, RMSE, R², and MAPE computed on the original day scale** (not the log scale — the whole point is to report metrics in units a stakeholder understands).
4. **`baseline_predict_log(y_train_log, n)`** — a baseline that predicts the training median for every row (robust to the tail, unlike the mean).
5. Train a `RandomForestRegressor(n_estimators=200, random_state=42)` on a 75/25 split (`random_state=42`) and compute `evaluate_regression` for both the model and the baseline.
6. **`test_model_beats_baseline(model_metrics, baseline_metrics)`** — an assert-based quality gate: raise `AssertionError` with a clear message if the model's MAE is not strictly better than the baseline's MAE.
7. **Outlier-impact analysis**: find the single test-set planet with the largest squared error, and report what percentage of the *total* test-set squared error that one point alone accounts for. Then, in 2–4 sentences, explain why RMSE/R² tell a different (more pessimistic) story than MAE here, and say which metric you'd actually lead with when reporting this model's quality to a non-technical stakeholder — and why.

## Starter cells

Imports and the raw data load are done for you. Function signatures are given — keep them as-is so the later cells (which call them) keep working. Fill in the TODO bodies.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RAW_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/planets.csv"
df_raw = pd.read_csv(RAW_URL)

print(df_raw.shape)
print(df_raw.isna().sum())
df_raw.head()

In [ ]:
def clean_planets(df: pd.DataFrame):
    """TODO:
    - Start an `audit` dict with audit["rows_in"] = len(df).
    - Drop rows where `orbital_period` is missing; record the count dropped
      as audit["dropped_missing_target"].
    - Drop rows where `distance` is missing; record the count dropped as
      audit["dropped_missing_distance"].
    - Record audit["mass_missing_fraction_remaining"] = the fraction of
      *remaining* rows where `mass` is still missing.
    - Drop the `mass` column entirely (do not impute it).
    - Add a `method_bucketed` column: the top 3 values of `method` by
      frequency kept as-is, everything else mapped to "Other".
    - Record audit["rows_out"] = len(df).
    - Return (df, audit).
    """
    raise NotImplementedError

In [ ]:
def build_features(df: pd.DataFrame):
    """TODO:
    - Build X by one-hot encoding `method_bucketed` (drop_first=True) and
      keeping `distance` and `year` as numeric columns.
    - Build y_log = log1p(orbital_period).
    - Return X, y_log.
    """
    raise NotImplementedError

In [ ]:
def evaluate_regression(y_true_log, y_pred_log) -> dict:
    """TODO:
    - Back-transform both y_true_log and y_pred_log with expm1 to get days.
    - Clip back-transformed predictions to be >= 0 (a model can predict a
      negative log-space value that expm1 turns into a tiny negative day
      count, which is nonsensical for a period).
    - Compute and return {"MAE_days": ..., "RMSE_days": ..., "R2": ...,
      "MAPE_pct": ...} using the *day-scale* values, not the log values.
    """
    raise NotImplementedError

In [ ]:
def baseline_predict_log(y_train_log, n: int):
    """TODO: return an array of length n, every entry equal to the
    median of y_train_log."""
    raise NotImplementedError

In [ ]:
# TODO:
# 1. df, audit = clean_planets(df_raw); print(audit)
# 2. X, y_log = build_features(df)
# 3. 75/25 train/test split of X, y_log with random_state=42
# 4. Fit RandomForestRegressor(n_estimators=200, random_state=42) on the
#    training data (log-space target) and predict on the test set.
# 5. baseline_log = baseline_predict_log(y_train_log, len(y_test_log))
# 6. model_metrics = evaluate_regression(y_test_log, pred_log)
#    baseline_metrics = evaluate_regression(y_test_log, baseline_log)
#    print both dicts.

In [ ]:
def test_model_beats_baseline(model_metrics: dict, baseline_metrics: dict):
    """TODO: assert model_metrics["MAE_days"] < baseline_metrics["MAE_days"],
    with an AssertionError message that reports both MAE values if it fails.
    Call it on your model_metrics/baseline_metrics and confirm it passes.
    """
    raise NotImplementedError

In [ ]:
# TODO: outlier-impact analysis
# 1. Back-transform y_test_log and pred_log to day-scale (clip predictions >= 0).
# 2. Compute the squared error for every test-set point.
# 3. Find the index of the single largest squared error, and print that
#    planet's true and predicted orbital period in days.
# 4. Print that point's squared error as a percentage of the *sum* of all
#    test-set squared errors.
# 5. Compare the MAE improvement (model vs. baseline, as a percentage) to
#    the R2 and RMSE you computed above -- do they tell the same story?

### Write-up

TODO: in 2–4 sentences, explain why RMSE and R² disagree with MAE on this dataset, and say which single metric you'd lead with when reporting this model to a non-technical stakeholder, and why.

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def clean_planets(df: pd.DataFrame):
    audit = {"rows_in": len(df)}

    df = df.dropna(subset=["orbital_period"]).copy()
    audit["dropped_missing_target"] = audit["rows_in"] - len(df)

    before = len(df)
    df = df.dropna(subset=["distance"]).copy()
    audit["dropped_missing_distance"] = before - len(df)

    audit["mass_missing_fraction_remaining"] = df["mass"].isna().mean()
    df = df.drop(columns=["mass"])

    top_methods = df["method"].value_counts().nlargest(3).index
    df["method_bucketed"] = np.where(df["method"].isin(top_methods), df["method"], "Other")

    audit["rows_out"] = len(df)
    return df, audit


def build_features(df: pd.DataFrame):
    X = pd.get_dummies(df[["distance", "year", "method_bucketed"]],
                        columns=["method_bucketed"], drop_first=True)
    y_log = np.log1p(df["orbital_period"])
    return X, y_log


def evaluate_regression(y_true_log, y_pred_log) -> dict:
    y_true = np.expm1(y_true_log)
    y_pred = np.clip(np.expm1(y_pred_log), 0, None)
    return {
        "MAE_days": mean_absolute_error(y_true, y_pred),
        "RMSE_days": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE_pct": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
    }


def baseline_predict_log(y_train_log, n: int):
    return np.full(n, np.median(y_train_log))


df, audit = clean_planets(df_raw)
print(audit)
# {'rows_in': 1035, 'dropped_missing_target': 43, 'dropped_missing_distance': 216,
#  'mass_missing_fraction_remaining': 0.358, 'rows_out': 776}

X, y_log = build_features(df)
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.25, random_state=42
)

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train_log)
pred_log = model.predict(X_test)

baseline_log = baseline_predict_log(y_train_log, len(y_test_log))

model_metrics = evaluate_regression(y_test_log, pred_log)
baseline_metrics = evaluate_regression(y_test_log, baseline_log)
print("model:", model_metrics)
print("baseline:", baseline_metrics)
# model:    {'MAE_days': 2226, 'RMSE_days': 22826, 'R2': -0.0007, 'MAPE_pct': 1110}
# baseline: {'MAE_days': 2364, 'RMSE_days': 22929, 'R2': -0.0097, 'MAPE_pct': 2390}


def test_model_beats_baseline(model_metrics: dict, baseline_metrics: dict):
    assert model_metrics["MAE_days"] < baseline_metrics["MAE_days"], (
        f"model MAE {model_metrics['MAE_days']:.1f} did not beat "
        f"baseline MAE {baseline_metrics['MAE_days']:.1f}"
    )

test_model_beats_baseline(model_metrics, baseline_metrics)  # passes

y_test_days = np.expm1(y_test_log)
pred_days = np.clip(np.expm1(pred_log), 0, None)
sq_err = (y_test_days - pred_days) ** 2
worst_idx = np.argmax(sq_err.values)
print("worst point: true =", y_test_days.iloc[worst_idx], "pred =", pred_days[worst_idx])
print("share of total squared error:", sq_err.iloc[worst_idx] / sq_err.sum())
# worst point: true = 318280.0 pred = 994.0
# share of total squared error: ~99.6%
```

**What this shows:** the model beats the median baseline on MAE by about 6% — a real, if modest, improvement on the typical planet. But RMSE and R² barely move (R² is essentially 0), because a single test-set planet with a true period of ~318,000 days (a very long-period, low-confidence measurement) contributes **99.6% of the total squared error** by itself. RMSE and R² are squared-error metrics, so one extreme outlier can drown out genuine improvement on every other point; MAE, being linear in the error, stays informative. For this dataset, MAE (and the improvement over baseline) is the honest number to lead with — quoting RMSE or R² alone would make a model that is actually helping on ~97% of planets look worthless, when the real story is "one extreme, high-uncertainty data point dominates the squared-error metrics."

</details>